In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Math

# ==============================================================================
# PROBLEM 10: Fully Symbolic Z-Transform via SymPy Definition Sum & Visualization
# Signal: Periodic impulse train x[n] = sum_{k=-inf}^{inf} delta[n - 4k]
# ==============================================================================

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Solution Overview (Fully Symbolic Calculation via SymPy)</b><br>
* <b>Signal:</b> Periodic impulse train x[n] = Σ δ[n - 4k] with period N = 4.<br>
* <b>Z-Transform Definition:</b> X(z) = Σ x[n] z<sup>-n</sup> evaluated over impulse sampling.<br>
* <b>Closed-Form Expression:</b> Expressed as a summation of powers of z<sup>-4</sup> or via Dirac comb properties.<br>
* <b>Note:</b> Use the slider below to dynamically change period N and observe the exact impulse locations and transform properties.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

# Define symbolic variables
n_sym = sp.Symbol('n', integer=True)
z_sym = sp.Symbol('z', complex=True)
k_sym = sp.Symbol('k', integer=True)
N_sym = sp.Symbol('N', integer=True, positive=True)

# 1. Symbolic Z-transform representation for periodic impulse train with period N=4
N_val_default = 4
sum_k = sp.Sum(z_sym**(-N_sym * k_sym), (k_sym, -sp.oo, sp.oo))

display(Math(f"X(z) = \\sum_{{k=-\\infty}}^{{\\infty}} z^{{-{N_val_default}k}}"))

def plot_problem_10(N_val):
    with out:
        clear_output(wait=True)
        
        fig, (ax_pz, ax_time) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [1, 2]})
        plt.subplots_adjust(wspace=0.25)

        # --- 1. Pole-Zero Map & ROC ---
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-2.0, 2.0)
        ax_pz.set_ylim(-2.0, 2.0)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        # ROC for periodic impulse train: Entire z-plane except z = 0 and z = inf
        x_vals = np.linspace(-2.5, 2.5, 400)
        y_vals = np.linspace(-2.5, 2.5, 400)
        X, Y = np.meshgrid(x_vals, y_vals)
        Z_dist = np.sqrt(X**2 + Y**2)
        roc_mask = (Z_dist > 0.05) & (Z_dist < 10.0)

        ax_pz.imshow(roc_mask, extent=(-2.5, 2.5, -2.5, 2.5), origin='lower', cmap='Greens', alpha=0.25, zorder=0)

        theta = np.linspace(0, 2*np.pi, 200)
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5)

        # Poles of X(z) are located at roots of z^N - 1 = 0 (Harmonic frequencies)
        k_indices = np.arange(N_val)
        pole_angles = 2 * np.pi * k_indices / N_val
        poles_x = np.cos(pole_angles)
        poles_y = np.sin(pole_angles)
        ax_pz.scatter(poles_x, poles_y, s=140, color='purple', marker='x', linewidths=3)

        ax_pz.set_title(f'Pole-Zero Map & ROC (Period N = {N_val})', fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)

        # Legend handles in a single straight line
        unit_circle_handle = plt.Line2D([0], [0], color='k', linestyle='--', alpha=0.5, label='Unit Circle')
        pole_handle = plt.Line2D([0], [0], marker='x', color='purple', markersize=8, markeredgewidth=3, linestyle='None', label=f'Poles (Roots of z^{N_val} = 1)')
        
        ax_pz.legend(handles=[unit_circle_handle, pole_handle], loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=8)

        # --- 2. Time Domain Plot ---
        n_vec = np.arange(-12, 13)
        x_n_vals = np.array([1 if n % N_val == 0 else 0 for n in n_vec])

        ax_time.stem(n_vec, x_n_vals, linefmt='r-', markerfmt='ro', basefmt='k-')
        ax_time.set_title(f'Temporal Evolution: Periodic Impulse Train (Period N = {N_val})', fontsize=10, fontweight='bold')
        ax_time.set_xlabel('Time index n', fontsize=9)
        ax_time.set_ylabel('x[n]', fontsize=9)
        ax_time.set_xlim(-13, 13)
        ax_time.set_ylim(-0.1, 1.4)
        ax_time.grid(True, linestyle=':', alpha=0.7)

        plt.show()

        # Explanation comments below the figures (in English for Springer submission)
        print("-" * 115)
        print("GEOMETRIC & THEORETICAL INTERPRETATION OF THE POLE-ZERO MAP:")
        print("1. Why are the poles located on the unit circle? Because the signal is an undamped periodic impulse train.")
        print("   Mathematically, the poles correspond to the roots of z^N = 1 (roots of unity), satisfying |z| = 1.")
        print("2. Since the periodic train repeats indefinitely without decaying or growing, its energy is distributed")
        print("   at harmonic frequencies, pushing the system into marginal stability where the Z-transform approaches infinity.")
        print("-" * 115)

# Slider for period N
N_slider = widgets.IntSlider(value=4, min=2, max=8, step=1, description='Period N:', style={'description_width': 'initial'})

plot_problem_10(N_slider.value)

interactive_plot = widgets.interactive(plot_problem_10, N_val=N_slider)
display(widgets.VBox([interactive_plot, out]))